# Домашняя работа 3

**ФИО и группа:** Кулик Георгий, Т26ОАД-Т.МО31.

## Подготовка

In [ ]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal

DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)

decisions = pd.read_csv(
    DATA / "decisions.csv",
    dtype={"lei": "string", "county_code": "string"},
)

## Исходники домашней работы

In [ ]:
org_keys = ["lei", "activity_year"]
long_keys = org_keys + ["action_taken"]
organizations = pd.read_csv(DATA / "organizations.csv", dtype={"lei": "string"})
lookup = organizations[org_keys + ["organization_name"]].copy()
assert not lookup.duplicated(org_keys).any()
print(organizations.head(3))
assert not organizations[org_keys].isna().any().any()
assert not decisions[org_keys].isna().any().any()
assert decisions["action_taken"].isin([1, 2, 3]).all()
display(decisions.head(4))

## H1 Профиль отказов

In [ ]:
prepared = decisions.copy()
prepared["is_denied"] = prepared["action_taken"].eq(3)
denial = prepared.groupby(org_keys).agg(
    n_decisions=("action_taken", "size"),
    n_denied=("is_denied", "sum"),
).reset_index()
denial["denial_rate"] = denial["n_denied"] / denial["n_decisions"]
home_denial_profile = denial.merge(
    lookup, on=org_keys, how="left", validate="one_to_one", indicator=True
)
assert len(home_denial_profile) == len(denial)
assert home_denial_profile["_merge"].eq("both").all()
assert home_denial_profile["n_decisions"].sum() == len(decisions)
assert home_denial_profile["n_denied"].sum() == decisions["action_taken"].eq(3).sum()
assert home_denial_profile["denial_rate"].between(0, 1).all()
home_denial_profile = home_denial_profile.drop(columns="_merge")
home_denial_profile.to_csv(WORK / "home_denial_profile.csv", index=False)
print("Организаций:", len(home_denial_profile))
print("Решений:", home_denial_profile["n_decisions"].sum())
print("Отказов:", home_denial_profile["n_denied"].sum())
print("Общая доля отказов:", home_denial_profile["n_denied"].sum() / home_denial_profile["n_decisions"].sum())
home_denial_profile.head()

## H2 Группы для просмотра

In [ ]:
large_groups = home_denial_profile.loc[
    home_denial_profile["n_decisions"].ge(20)
].copy()
large_groups.to_csv(WORK / "home_denial_groups20.csv", index=False)
large_total = large_groups["n_decisions"].sum()
large_denied = large_groups["n_denied"].sum()
print("Групп с 20+ решениями:", len(large_groups))
print("Решений в группах:", large_total)
print("Отказов в группах:", large_denied)
print("Общая доля отказов выбранных групп:", large_denied / large_total)
print("Простое среднее долей отказов организаций:", large_groups["denial_rate"].mean())
large_groups.head()
assert large_groups["n_decisions"].ge(20).all()
assert home_denial_profile["n_decisions"].sum() == len(decisions)
print("Всего групп:", len(home_denial_profile), "Осталось:", len(large_groups))
print("Общая доля отказов полного профиля:", home_denial_profile["n_denied"].sum() / home_denial_profile["n_decisions"].sum())
print("Простое среднее долей полного профиля:", home_denial_profile["denial_rate"].mean())

## H3 Длинная, широкая и снова длинная форма

In [ ]:
long = decisions.groupby(long_keys).size().rename("n").reset_index()
wide = long.pivot(index=org_keys, columns="action_taken", values="n")
wide = wide.reindex(columns=[1, 2, 3], fill_value=0).fillna(0).astype("int64")
wide.columns.name = None
wide = wide.rename(columns={1:"issued", 2:"approved_not_accepted", 3:"denied"}).reset_index()
value_cols = ["issued", "approved_not_accepted", "denied"]
roundtrip = wide.melt(
    id_vars=org_keys, value_vars=value_cols,
    var_name="outcome", value_name="n"
)
roundtrip["action_taken"] = roundtrip["outcome"].map({
    "issued": 1, "approved_not_accepted": 2, "denied": 3
}).astype("int64")
roundtrip = roundtrip.loc[roundtrip["n"].gt(0), long_keys + ["n"]]
roundtrip = roundtrip.sort_values(long_keys).reset_index(drop=True)
expected = long.sort_values(long_keys).reset_index(drop=True)
assert_frame_equal(roundtrip, expected, check_exact=True)
assert roundtrip['n'].sum() == len(decisions)
assert wide[value_cols].sum().sum() == len(decisions)
print("Round-trip совпал с исходной сводкой.")
print("Строк в исходной long-сводке:", len(long))
print("Строк после обратного преобразования:", len(roundtrip))
# Нули — отсутствующие сочетания; они удалены перед сравнением с исходной long-сводкой.

## H+ Безопасное соединение

In [ ]:
def safe_enrich(left, right):
    if left[org_keys].isna().any().any() or right[org_keys].isna().any().any():
        raise ValueError("В ключах соединения есть пропуски")
    if right.duplicated(org_keys).any():
        raise ValueError("Ключи справочника повторяются")
    result = left.merge(right, on=org_keys, how="left", validate="many_to_one", indicator=True)
    assert len(result) == len(left)
    assert_frame_equal(result[left.columns].reset_index(drop=True), left.reset_index(drop=True), check_exact=True)
    return result, result.loc[result["_merge"].eq("left_only")].copy()

full_lookup = lookup.copy()
partial_lookup = lookup.iloc[1:].copy()
replayed_lookup = pd.concat([lookup, lookup.iloc[[0]]], ignore_index=True)
conflicting_lookup = pd.concat([lookup, lookup.iloc[[0]].assign(organization_name="Другое название")], ignore_index=True)

safe_full, unmatched_full = safe_enrich(decisions, full_lookup)
assert unmatched_full.empty
safe_partial, unmatched_partial = safe_enrich(decisions, partial_lookup)
assert len(safe_partial) == len(decisions)
assert set(unmatched_partial["source_row"]) == set(decisions["source_row"]) - set(decisions.merge(partial_lookup, on=org_keys, how="inner")["source_row"])
print("Полный справочник: несовпадений", len(unmatched_full))
print("Неполный справочник: несовпадений", len(unmatched_partial))

name_counts = conflicting_lookup.groupby(org_keys)["organization_name"].nunique(dropna=False)
conflicts = name_counts[name_counts.gt(1)]
assert not conflicts.empty
print("Ключи с конфликтующими названиями:")
print(conflicts)

missing_left = decisions.copy()
missing_left.loc[missing_left.index[0], "lei"] = pd.NA
missing_right = lookup.copy()
missing_right.loc[missing_right.index[0], "lei"] = pd.NA
for label, left, right in [
    ("Повтор ключа", decisions, replayed_lookup),
    ("Конфликт имён", decisions, conflicting_lookup),
    ("Пропуск слева", missing_left, full_lookup),
    ("Пропуск справа", decisions, missing_right),
]:
    try:
        safe_enrich(left, right)
    except ValueError as error:
        print(label, "— остановлено:", error)
    else:
        raise AssertionError(f"Проверка не остановила случай: {label}")